# Quadratic Estimator Tutorial

This notebook demonstrates how to use the quadratic estimator functionality in lenspyx to reconstruct the CMB lensing potential from simulated lensed CMB maps.

The tutorial covers:
1. Generating lensed CMB maps
2. Setting up inverse-variance filtering
3. Applying quadratic estimators
4. Normalizing and analyzing the results

The computations in this notebook should take $\mathcal{O}$(minutes) on a modern laptop.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from lenspyx import synfast, get_geom
from lenspyx.utils import get_ffp10_cls
from lenspyx.utils_hp import gauss_beam, almxfl, alm2cl, alm_copy, synalm
from lenspyx.qest.qest import Qlms, OpFilt

## Step 1: Set up parameters

First, we'll set up the parameters for our simulation and reconstruction.

In [ ]:
# Parameters
lmax_unl = 3000  # Maximum multipole for unlensed fields
lmax_filt = 2000  # Maximum multipole for filtering
lmax_qlm = 500   # Maximum multipole for QE output
geom_info = ('thingauss', {'lmax': 4000, 'smax': 2})  # Geometry specification

## Step 2: Get CMB power spectra

We'll use the FFP10 (Planck Full Focal Plane simulation 10) power spectra.

In [ ]:
# Get Cls from FFP10
cls_unl, cls_len, cls_glen = get_ffp10_cls(lmax=lmax_unl)
geom = get_geom(geom_info)

# Plot the power spectra
ls = np.arange(2, 3000)
plt.figure(figsize=(10, 6))
plt.loglog(ls, ls * (ls + 1) * cls_unl['tt'][ls] / (2 * np.pi), label='TT (unlensed)')
plt.loglog(ls, ls * (ls + 1) * cls_len['tt'][ls] / (2 * np.pi), label='TT (lensed)')
plt.loglog(ls, ls * (ls + 1) * cls_unl['ee'][ls] / (2 * np.pi), label='EE (unlensed)')
plt.loglog(ls, ls * (ls + 1) * cls_len['ee'][ls] / (2 * np.pi), label='EE (lensed)')
plt.loglog(ls, ls * (ls + 1) * cls_unl['bb'][ls] / (2 * np.pi), label='BB (unlensed)')
plt.loglog(ls, ls * (ls + 1) * cls_len['bb'][ls] / (2 * np.pi), label='BB (lensed)')
plt.loglog(ls, ls * (ls + 1) * cls_unl['te'][ls] / (2 * np.pi), label='TE (unlensed)')
plt.loglog(ls, ls * (ls + 1) * cls_len['te'][ls] / (2 * np.pi), label='TE (lensed)')
plt.xlabel('Multipole $\ell$')
plt.ylabel('$\ell(\ell+1)C_\ell/(2\pi)$ [$\mu K^2$]')
plt.title('CMB Power Spectra')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Step 3: Define beam and noise properties

We'll define a simple Gaussian beam and white noise for our simulation.

In [ ]:
# Define beam and noise properties
beam = gauss_beam(5. / 180 / 60 * np.pi, lmax=lmax_filt)  # 5 arcmin beam
inoise = {
    'tt': beam ** 2 / (35. / 180 / 60 * np.pi) ** 2,  # 35 μK-arcmin for temperature
    'ee': beam ** 2 / (55. / 180 / 60 * np.pi) ** 2,  # 55 μK-arcmin for polarization
    'bb': beam ** 2 / (55. / 180 / 60 * np.pi) ** 2   # 55 μK-arcmin for polarization
}
transfs = {f: np.ones(lmax_filt + 1, dtype=float) for f in 'teb'}  # Transfer functions

# Plot the beam and noise
ls = np.arange(2, lmax_filt)
plt.figure(figsize=(10, 6))
plt.semilogy(ls, beam[ls]**2, label='Beam (5 arcmin)')
plt.semilogy(ls, 1.0 / inoise['tt'][ls], label='TT Noise (35 μK-arcmin)')
plt.semilogy(ls, 1.0 / inoise['ee'][ls], label='EE/BB Noise (55 μK-arcmin)')
plt.xlabel('Multipole $\ell$')
plt.ylabel('Power')
plt.title('Beam and Noise Properties')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Step 4: Generate lensed CMB maps

Now we'll generate lensed CMB maps using the `synfast` function.

In [ ]:
# Generate lensed CMB maps
maps, (unl_alms, unl_lab) = synfast(cls_unl, lmax=lmax_unl, geometry=geom_info, verbose=True, alm=True)

## Step 5: Convert maps to harmonic space

We'll convert the maps to harmonic space for further processing.

In [ ]:
# Convert maps to harmonic space
tlm = geom.adjoint_synthesis(maps['T'], 0, lmax_filt, lmax_filt, 0).squeeze()
eblm = geom.adjoint_synthesis(maps['QU'], 2, lmax_filt, lmax_filt, 0)

## Step 6: Apply transfer functions and add noise

We'll apply the transfer functions and add instrumental noise to the maps.

In [ ]:
# Apply transfer functions
almxfl(tlm, transfs['t'], lmax_filt, True)
almxfl(eblm[0], transfs['e'], lmax_filt, True)
almxfl(eblm[1], transfs['b'], lmax_filt, True)

# Add instrumental noise
tlm_noisy = tlm.copy() + synalm(1. / inoise['tt'], lmax_filt, lmax_filt)
elm_noisy = eblm[0].copy() + synalm(1. / inoise['ee'], lmax_filt, lmax_filt)
blm_noisy = eblm[1].copy() + synalm(1. / inoise['bb'], lmax_filt, lmax_filt)
alms = {'t': tlm_noisy, 'e': elm_noisy, 'b': blm_noisy}

# Get input lensing potential for comparison
plm_in = alm_copy(unl_alms[unl_lab.index('p')], lmax_unl, lmax_qlm, lmax_qlm)

## Step 7: Create inverse-variance filtering object

Now we'll create an inverse-variance filtering object for the quadratic estimator.

In [ ]:
# Helper function to copy cls for the desired fields
def copy_cls(cls, include=()):
    """Returns cls for the desired fields only"""
    ret = {}
    for k in cls:
        if k[0] in include and k[1] in include:
            ret[k] = np.copy(cls[k])
    return ret

## Step 8: Apply different quadratic estimators

We'll apply different types of quadratic estimators and compare their performance.

In [ ]:
# Apply different quadratic estimators
results = {}
for qe_key, qe_lab in zip(['ptt', 'p_p', 'p'], ['TT', 'Pol', 'GMV']):
    print(f"\nApplying {qe_lab} estimator...")
    
    # Select fields to include based on estimator type
    includes = ['t'] * (qe_key in ['ptt', 'p']) + ['e', 'b'] * (qe_key in ['p_p', 'p'])
    cls_filt = copy_cls(cls_len, include=includes)
    
    # Create inverse-variance filtering object
    filtr = OpFilt(cls_filt, transfs, inoise)
    
    # Create QE calculator object
    qlms_dd = Qlms(filtr, filtr, cls_len, lmax_qlm)
    
    # Calculate (unormalized) lensing potentials (gradient and curl)
    plm, olm = qlms_dd.get_qlms(qe_key, alms, verbose=True)
    
    # Calculate estimator normalization
    rp, ro = qlms_dd.get_response(qe_key, 'p', cls_len)
    
    # Store results
    results[qe_key] = {
        'plm': plm.copy(),  # Store a copy to preserve the original
        'olm': olm.copy() if olm is not None else None,
        'rp': rp,
        'ro': ro,
        'label': qe_lab
    }
    
    # Apply normalization to get the normalized lensing potential
    plm_norm = plm.copy()
    # Avoid division by zero for L=0,1
    rp_safe = rp.copy()
    rp_safe[:2] = 1.0
    almxfl(plm_norm, 1.0 / rp_safe, lmax_qlm, True)
    
    # Calculate correlation with input
    ls = np.arange(10, 100)  # Focus on a reasonable L range
    corr = alm2cl(plm_norm, plm_in, lmax_qlm, lmax_qlm, lmax_qlm)[ls]
    auto = alm2cl(plm_norm, plm_norm, lmax_qlm, lmax_qlm, lmax_qlm)[ls]
    input_cl = cls_unl['pp'][ls]
    
    # Calculate correlation coefficient
    corr_coeff = corr / np.sqrt(auto * input_cl)
    
    print(f"  Mean correlation coefficient (L=10-100): {np.mean(corr_coeff):.3f}")

## Step 9: Plot results

Finally, we'll plot the results to compare the different estimators.

In [ ]:
# Plot results
ls = np.arange(2, lmax_qlm + 1)
wls = ls ** 2 * (ls + 1) ** 2 * 1e7 / (2 * np.pi)  # Scaling for plotting

plt.figure(figsize=(12, 8))

# Plot input theory spectrum
plt.plot(ls, wls * cls_unl['pp'][ls], c='k', label=r'$C_L^{\phi\phi}$ (Theory)', linewidth=2)

# Plot results for each estimator
colors = {'ptt': 'b', 'p_p': 'r', 'p': 'g'}
for qe_key, res in results.items():
    plm, rp = res['plm'], res['rp']
    label = res['label']
    
    # Plot auto-spectrum
    auto = wls / rp[ls] ** 2 * alm2cl(plm, plm, lmax_qlm, lmax_qlm, lmax_qlm)[ls]
    plt.plot(ls, auto, color=colors[qe_key], label=f'{label} Auto', linewidth=1.5)
    
    # Plot N0 bias
    plt.plot(ls, wls / rp[ls], ls='--', color=colors[qe_key], label=f'{label} $N^0_L$', linewidth=1.5)

plt.xlabel(r'Multipole $L$', fontsize=14)
plt.ylabel(r'$10^7\cdot L^2(L + 1)^2 C_L^{\phi\phi} / 2\pi$', fontsize=14)
plt.legend(fontsize=12)
plt.xscale('log')
plt.yscale('log')
plt.xlim(2, lmax_qlm)
plt.ylim(0.1, 100)
plt.grid(True, alpha=0.3)
plt.title('Quadratic Estimator Performance Comparison', fontsize=16)
plt.tight_layout()
plt.show()

## Conclusion

In this tutorial, we've demonstrated how to use the quadratic estimator functionality in lenspyx to reconstruct the CMB lensing potential from simulated lensed CMB maps. We've compared three different types of estimators:

1. **Temperature-only (TT)**: Uses only temperature information
2. **Polarization-only (Pol)**: Uses only E and B mode polarization
3. **Minimum-variance (GMV)**: Combines temperature and polarization optimally

As expected, the minimum-variance estimator provides the best performance, achieving the highest correlation with the input lensing potential.

For more advanced usage, including custom filtering schemes and noise bias subtraction, please refer to the example scripts in the `lenspyx/tests/qes` directory.